# 5 - Silver

Preparar os dados da camada Bronze para análise, realizando limpeza, padronização de tipos, validações de qualidade e remoção de duplicidades.

**5.1 - Leitura da Bronze**

In [0]:
from pyspark.sql import functions as F
CATALOG="main"; SCHEMA="students_performance"
BRONZE_TABLE=f"{CATALOG}.{SCHEMA}.students_performance_bronze"
SILVER_TABLE=f"{CATALOG}.{SCHEMA}.students_performance_silver"
df_bronze=spark.table(BRONZE_TABLE)


**5.2 - Padronização dos dados**

Os campos textuais passam por tratamento com TRIM, eliminando espaços desnecessários.

In [0]:
df_silver = (
    df_bronze
        .withColumn("gender", F.trim(F.col("gender")))
        .withColumn("race_ethnicity", F.trim(F.col("race_ethnicity")))
        .withColumn(
            "parental_level_of_education",
            F.trim(F.col("parental_level_of_education"))
        )
        .withColumn("lunch", F.trim(F.col("lunch")))
        .withColumn(
            "test_preparation_course",
            F.trim(F.col("test_preparation_course"))
        )
        .withColumn("math_score", F.col("math_score").cast("int"))
        .withColumn("reading_score", F.col("reading_score").cast("int"))
        .withColumn("writing_score", F.col("writing_score").cast("int"))
        .dropDuplicates()
)


**5.3 - Tratamento de duplicidades**

Momento para eliminar registros completamente duplicados.

In [0]:
valid_genders=["female","male"]
valid_races=[f"group {x}" for x in ["A","B","C","D","E"]]
valid_education=["some high school","high school","some college","associate's degree","bachelor's degree","master's degree"]
valid_lunch=["standard","free/reduced"]; valid_preparation=["none","completed"]
df_silver=(df_silver
 .withColumn("dq_complete_record",F.col("gender").isNotNull()&F.col("race_ethnicity").isNotNull()&F.col("parental_level_of_education").isNotNull()&F.col("lunch").isNotNull()&F.col("test_preparation_course").isNotNull()&F.col("math_score").isNotNull()&F.col("reading_score").isNotNull()&F.col("writing_score").isNotNull())
 .withColumn("dq_valid_scores",F.col("math_score").between(0,100)&F.col("reading_score").between(0,100)&F.col("writing_score").between(0,100))
 .withColumn("dq_valid_domains",F.col("gender").isin(valid_genders)&F.col("race_ethnicity").isin(valid_races)&F.col("parental_level_of_education").isin(valid_education)&F.col("lunch").isin(valid_lunch)&F.col("test_preparation_course").isin(valid_preparation))
 .withColumn("dq_valid_record",F.col("dq_complete_record")&F.col("dq_valid_scores")&F.col("dq_valid_domains"))
 .withColumn("average_score",(F.col("math_score")+F.col("reading_score")+F.col("writing_score"))/F.lit(3.0)))


**5.4 - Validação de qualidade**

Momento para verificar se os registros estão adequados.

In [0]:
quality_report=df_silver.agg(F.count("*").alias("total_records"),F.sum(F.col("dq_complete_record").cast("int")).alias("complete_records"),F.sum((~F.col("dq_valid_scores")).cast("int")).alias("invalid_score_records"),F.sum((~F.col("dq_valid_domains")).cast("int")).alias("invalid_domain_records"),F.sum(F.col("dq_valid_record").cast("int")).alias("valid_records"))
display(quality_report)
(df_silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(SILVER_TABLE))
print(f"Silver armazenada em: {SILVER_TABLE}")


total_records,complete_records,invalid_score_records,invalid_domain_records,valid_records
1000,1000,0,0,1000


Silver armazenada em: main.students_performance.students_performance_silver
